# Ab Initio Molecular Dynamics

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PhilipVinc/ComputationalQuantumPhysics/blob/main/Notebooks/6-Elecs/2-AIMD.ipynb)

In the previous notebook we solved the electronic Schrödinger equation for **fixed** nuclei, obtaining the total energy $E(\mathbf{R})$ as a function of nuclear positions. This energy defines a **potential energy surface** (PES).

Today we combine the electronic structure solver with classical mechanics to simulate **nuclear motion**:

1. **Part 1** — Compute forces on nuclei ($\mathbf{F} = -\nabla_{\mathbf{R}} E$) via finite differences, and visualise them as a vector field.
2. **Part 2** — Integrate Newton’s equations of motion using Velocity Verlet to watch atoms find their equilibrium geometry.

We work in **atomic units** throughout: distances in Bohr ($a_0 = 0.529\,\text{Å}$), energies in Hartree, masses in $m_e$, and time in $\hbar/E_h \approx 0.024\,\text{fs}$.

In [ ]:
# !pip install pyscf   # uncomment if needed

import numpy as np
import matplotlib.pyplot as plt
from pyscf import gto, scf

MASS_H = 1836.15267  # proton mass in electron masses (atomic units)
BOHR_TO_ANG = 0.529177  # 1 Bohr in Angstrom

---
## Part 1 — Forces on Nuclei

Within the Born–Oppenheimer approximation the nuclei move on the PES $E(\mathbf{R})$.
The force on nucleus $i$ along Cartesian direction $\alpha$ is

$$F_{i\alpha} = -\frac{\partial E}{\partial R_{i\alpha}}.$$

We will first compute this derivative numerically with **central finite differences**, then compare with PySCF’s built-in **analytical gradient**.

### Task 1 — Energy as a function of nuclear positions

Write a function that, given an array of nuclear positions (all hydrogen) and a basis set, returns the RHF total energy. You already know all the ingredients from notebook 1: build a `gto.Mole`, run `scf.RHF`, return the energy.

**Hints:**
- Build the atom string as `"H x1 y1 z1; H x2 y2 z2; ..."`.
- Set `mol.unit = 'Bohr'` (we work in atomic units).
- Set `mol.verbose = 0` to suppress output.

In [ ]:
def compute_energy(positions, basis='sto-3g'):
    """
    Compute the RHF total energy for hydrogen atoms.

    Parameters
    ----------
    positions : ndarray, shape (n_atoms, 3)
        Nuclear coordinates in Bohr.
    basis : str
        Gaussian basis set name.

    Returns
    -------
    energy : float
        Total RHF energy in Hartree.
    """
    # --- your code here ---
    pass

In [ ]:
# --- Test your function ---
pos_eq = np.array([[0., 0., 0.], [0., 0., 1.4]])  # H2 at ~0.74 Angstrom
E = compute_energy(pos_eq)
print(f"E(H2, R=1.4 Bohr) = {E:.6f} Ha")
assert abs(E - (-1.1168)) < 0.01, f"Energy looks wrong: {E}"
print("Passed!")

### Task 2 — Forces via finite differences

The **central finite-difference** approximation to the derivative is

$$F_{i\alpha} = -\frac{\partial E}{\partial R_{i\alpha}} \approx -\frac{E(\mathbf{R} + h\,\hat{e}_{i\alpha}) - E(\mathbf{R} - h\,\hat{e}_{i\alpha})}{2h}$$

where $\hat{e}_{i\alpha}$ is the unit vector that displaces atom $i$ along direction $\alpha \in \{x,y,z\}$, and $h$ is a small step size (e.g. $h = 10^{-3}$ Bohr). The error is $\mathcal{O}(h^2)$.

**Task:** Implement `compute_forces_fd` below. Loop over all atoms and all three Cartesian directions. For each $(i,\alpha)$, shift the positions by $\pm h$ and evaluate the energy twice.

In [ ]:
def compute_forces_fd(positions, basis='sto-3g', h=1e-3):
    """
    Compute nuclear forces using central finite differences.

    Parameters
    ----------
    positions : ndarray, shape (n_atoms, 3)
        Nuclear coordinates in Bohr.
    basis : str
        Gaussian basis set name.
    h : float
        Finite-difference step size in Bohr.

    Returns
    -------
    forces : ndarray, shape (n_atoms, 3)
        Forces in Hartree/Bohr.
    """
    n_atoms = positions.shape[0]
    forces = np.zeros_like(positions)

    for i in range(n_atoms):
        for alpha in range(3):
            # --- your code here ---
            # 1. Create pos_plus = positions shifted by +h along (i, alpha)
            # 2. Create pos_minus = positions shifted by -h along (i, alpha)
            # 3. Evaluate energies at both configurations
            # 4. Apply the central-difference formula
            pass

    return forces

#### Checking your result: PySCF analytical gradient

PySCF can compute the **exact** gradient of the RHF energy analytically (no finite-difference error). This uses the formula

$$\frac{\partial E_{\text{RHF}}}{\partial R_{i\alpha}} = \text{(Hellmann–Feynman + Pulay terms)}$$

which is implemented via `mf.nuc_grad_method().kernel()`. We provide the wrapper below. Use it to validate your finite-difference forces.

In [ ]:
def compute_forces_analytical(positions, basis='sto-3g'):
    """Compute forces using PySCF's analytical RHF gradient."""
    atom_str = '; '.join(f'H {x} {y} {z}' for x, y, z in positions)
    mol = gto.Mole()
    mol.atom = atom_str
    mol.basis = basis
    mol.unit = 'Bohr'
    mol.verbose = 0
    mol.build()
    mf = scf.RHF(mol)
    mf.verbose = 0
    mf.kernel()
    g = mf.nuc_grad_method()
    g.verbose = 0
    return -g.kernel()  # force = -gradient

In [ ]:
# --- Compare finite-difference vs analytical forces at a test configuration ---
pos_test = np.array([[0.0, 0.0, 0.0], [0.3, 0.5, 1.8]])

F_fd   = compute_forces_fd(pos_test)
F_anal = compute_forces_analytical(pos_test)

print("Finite-difference forces (Ha/Bohr):")
print(F_fd)
print("\nAnalytical forces (Ha/Bohr):")
print(F_anal)
print(f"\nMax absolute difference: {np.max(np.abs(F_fd - F_anal)):.2e}")

### Visualising the force field

Fix one hydrogen atom at the origin and move the second atom through a grid of positions. At each grid point, compute the force on the mobile atom. This gives a **vector field** $\mathbf{F}(\mathbf{r})$.

In principle, the data is a 3-vector at every point in a 3D box, i.e. an array of shape `(nx, ny, nz, 3)`. For visualisation we compute a 2D slice in the $xz$-plane ($y=0$), stored as `(nz, nx, 3)`.

**Task:** Write a loop to fill in `forces_grid` and `energy_grid` using your `compute_forces_fd` function. 
Then plot the resulting force field using a 'quiver' plot (which shows the arrows pointing in the direction of the vector field).

For simplicity we will assume that one atom is fixed at (0,0,0) and will only consider the force acting on the other atom which is moved around.

Plot the 'exact finite difference' vector field as well as the 'analytical one' and compare. in particular look at what different values of the finite difference h does to the accuracy.

In [ ]:
# --- Grid setup ---
nx, nz = 10, 10
x_vals = np.linspace(-3.0, 3.0, nx)
z_vals = np.linspace(-3.0, 3.0, nz)
x_grid, z_grid = np.meshgrid(x_vals, z_vals)  # shape (nz, nx)

pos_fixed = np.array([0.0, 0.0, 0.0])  # atom 1 at origin
min_dist = 0.8  # skip points closer than this to avoid singularity

# Storage arrays
forces_fd_grid   = np.full((nz, nx, 3), np.nan)
forces_anal_grid = np.full((nz, nx, 3), np.nan)
energy_grid      = np.full((nz, nx), np.nan)


In [ ]:
def plot_force_field_2d(x_grid, z_grid, forces, energy=None, ax=None, title=""):
    """
    Plot a 2D force vector field in the xz-plane.

    Parameters
    ----------
    x_grid, z_grid : 2D arrays, shape (nz, nx) from meshgrid.
    forces : 3D array, shape (nz, nx, 3) — force vectors (Fx, Fy, Fz).
        Only the x and z components are plotted.
    energy : 2D array, shape (nz, nx), optional — for contour background.
    ax : matplotlib Axes, optional.
    title : str
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))

    Fx = forces[:, :, 0]
    Fz = forces[:, :, 2]
    mag = np.sqrt(Fx**2 + Fz**2)

    # Energy contour background
    if energy is not None:
        # plot a countour plot of the energy with a few lvels

    # Arrows: normalised to unit length, coloured by magnitude (log scale)
    valid = ~np.isnan(mag) & (mag > 0)
    # plot the arrows with a quiver plot


In [ ]:
# --- Plot the force fields side by side ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_force_field_2d(x_grid, z_grid, forces_fd_grid, energy=energy_grid,
                    ax=axes[0], title='Finite differences')
plot_force_field_2d(x_grid, z_grid, forces_anal_grid, energy=energy_grid,
                    ax=axes[1], title='Analytical gradient')

plt.tight_layout()
plt.show()

**Remarks::**
- In which direction do the arrows point at large distance? At short distance? Where are they zero?
- How many energy evaluations does the finite-difference method need per grid point? How does this scale with the number of atoms?
- What happens if you change the step size $h$? Try $h = 10^{-2}$ and $h = 10^{-5}$.

---
## Part 2 — Molecular Dynamics

We now treat the nuclei as classical particles obeying Newton’s equations:

$$m_i \ddot{\mathbf{R}}_i = \mathbf{F}_i = -\nabla_{\mathbf{R}_i} E(\mathbf{R})$$

At each time step we:
1. Solve the electronic problem to get $E(\mathbf{R})$ and $\mathbf{F}(\mathbf{R})$.
2. Advance the nuclear positions and velocities.

This is called **Born–Oppenheimer molecular dynamics** (BOMD).

### The Velocity Verlet integrator

Given positions $\mathbf{R}(t)$, velocities $\mathbf{v}(t)$, and forces $\mathbf{F}(t)$, one step of size $\Delta t$ proceeds as:

$$\mathbf{R}(t+\Delta t) = \mathbf{R}(t) + \mathbf{v}(t)\,\Delta t + \frac{\mathbf{F}(t)}{2m}\,\Delta t^2$$

$$\mathbf{F}(t+\Delta t) = \mathbf{F}\bigl(\mathbf{R}(t+\Delta t)\bigr) \quad \text{(recompute forces at new positions)}$$

$$\mathbf{v}(t+\Delta t) = \mathbf{v}(t) + \frac{\mathbf{F}(t) + \mathbf{F}(t+\Delta t)}{2m}\,\Delta t$$

Velocity Verlet is **symplectic** (preserves phase-space volume), **time-reversible**, and conserves energy to $\mathcal{O}(\Delta t^2)$.

For the time step, the fastest motion is H–H stretching (~4400 $cm^{-1}$, period $\approx$ 310 a.u.). We need $\Delta t$ well below this, so $\Delta t = 20$ a.u. ($\approx 0.5$ fs) is a safe choice.

### Task 3 — Implement the Velocity Verlet step

Complete the function below. It takes the current state `(positions, velocities, forces)`, advances by one time step $\Delta t$, and returns the new state.

**Important:** The function receives `force_fn`, a callable that takes positions and returns forces. Use it to compute $\mathbf{F}(t+\Delta t)$ after updating positions.

We use the analytical gradient for speed (since the MD loop calls the force function hundreds of times).

In [ ]:
def velocity_verlet_step(positions, velocities, forces, dt, mass, force_fn):
    """
    Perform one Velocity Verlet integration step.

    Parameters
    ----------
    positions  : ndarray (n_atoms, 3)  — current positions in Bohr
    velocities : ndarray (n_atoms, 3)  — current velocities in Bohr/(a.u. time)
    forces     : ndarray (n_atoms, 3)  — current forces in Ha/Bohr
    dt         : float                 — time step in a.u.
    mass       : float                 — atomic mass in m_e
    force_fn   : callable              — positions -> forces

    Returns
    -------
    new_positions, new_velocities, new_forces
    """
    # --- your code here ---
    # 1. Update positions:    R(t+dt) = R(t) + v(t)*dt + F(t)/(2m)*dt^2
    # 2. Compute new forces:  F(t+dt) = force_fn(new_positions)
    # 3. Update velocities:   v(t+dt) = v(t) + (F(t) + F(t+dt))/(2m)*dt
    pass

In [ ]:
def run_md(positions_init, velocities_init, mass, dt, n_steps, force_fn):
    """
    Run a molecular dynamics simulation using Velocity Verlet.

    Returns
    -------
    trajectory : ndarray (n_steps+1, n_atoms, 3)
    energies   : ndarray (n_steps, 3) — columns are (KE, PE, Total)
    """
    positions  = positions_init.copy()
    velocities = velocities_init.copy()
    forces     = force_fn(positions)

    trajectory = [positions.copy()]
    energies   = []

    for step in range(n_steps):
        positions, velocities, forces = velocity_verlet_step(
            positions, velocities, forces, dt, mass, force_fn
        )
        trajectory.append(positions.copy())

        ke = 0.5 * mass * np.sum(velocities**2)
        pe = compute_energy(positions)
        energies.append([ke, pe, ke + pe])

        if (step + 1) % 50 == 0:
            print(f"  step {step+1:4d}/{n_steps}  E_tot = {ke+pe:.6f} Ha")

    return np.array(trajectory), np.array(energies)


def plot_md_results(trajectory, energies, dt):
    """Plot bond distances and energies from an MD trajectory."""
    n_steps = len(energies)
    n_atoms = trajectory.shape[1]
    time = np.arange(1, n_steps + 1) * dt

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # --- Panel 1: pairwise distances ---
    for i in range(n_atoms):
        for j in range(i + 1, n_atoms):
            d = np.linalg.norm(trajectory[1:, i] - trajectory[1:, j], axis=1)
            axes[0].plot(time, d, label=f'd({i+1},{j+1})')
    axes[0].axhline(1.4, color='k', ls='--', lw=0.8, label='$R_{eq}$ (1.4 Bohr)')
    axes[0].set_xlabel('Time (a.u.)')
    axes[0].set_ylabel('Distance (Bohr)')
    axes[0].legend(fontsize=8)
    axes[0].set_title('Bond distances')

    # --- Panel 2: energies ---
    axes[1].plot(time, energies[:, 0], label='KE')
    axes[1].plot(time, energies[:, 1], label='PE')
    axes[1].plot(time, energies[:, 2], label='Total', color='k', lw=1.5)
    axes[1].set_xlabel('Time (a.u.)')
    axes[1].set_ylabel('Energy (Ha)')
    axes[1].legend()
    axes[1].set_title('Energy conservation')

    plt.tight_layout()
    plt.show()

### Task 4 — H$_2$ molecular dynamics

Run a few simulations of two hydrogen atoms starting from **random initial positions** (within a box of $\pm 2$ Bohr) and zero initial velocity.

1. Generate random starting positions. Ensure the atoms are at least 0.8 Bohr apart.
2. Run the MD for $\sim$300–500 steps with $\Delta t = 20$ a.u.
3. Use `plot_md_results` to visualise bond distance and energy vs time.
4. Repeat for 2–3 different random seeds.

**What to observe:**
- The atoms should oscillate around the equilibrium distance ($\approx 1.4$ Bohr).
- Total energy should be conserved (to machine precision for small $\Delta t$).
- The system does **not** truly “equilibrate” — in Newtonian mechanics without dissipation, the atoms oscillate forever. True thermalisation requires coupling to a heat bath (see the extension below).

In [ ]:
# --- Task 4: H2 simulations ---

dt = 20.0       # time step in a.u. (~0.5 fs)
n_steps = 400

# Example: generate random initial positions
rng = np.random.default_rng(seed=42)

def random_h2_positions(rng, box_size=2.0, min_dist=0.8):
    """Generate random positions for 2 H atoms, separated by at least min_dist."""
    while True:
        pos = rng.uniform(-box_size, box_size, size=(2, 3))
        if np.linalg.norm(pos[0] - pos[1]) >= min_dist:
            return pos

# --- your code here ---
# pos_init = random_h2_positions(rng)
# vel_init = np.zeros((2, 3))
# traj, ener = run_md(pos_init, vel_init, MASS_H, dt, n_steps, compute_forces_analytical)
# plot_md_results(traj, ener, dt)

### Task 5 — Four hydrogen atoms

Repeat the simulation with **4 hydrogen atoms** starting from random positions in a box of $\pm 3$ Bohr. Observe how they pair up into H$_2$ molecules.

- You may need more steps ($\sim$800–1000) for the atoms to find each other.
- Plot all six pairwise distances. Which pairs end up bonded?
- Does the final pairing depend on the initial configuration?

In [ ]:
# --- Task 5: H4 simulations ---

def random_h_positions(rng, n_atoms=4, box_size=3.0, min_dist=0.8):
    """Generate random positions for n_atoms H, all pairwise distances >= min_dist."""
    while True:
        pos = rng.uniform(-box_size, box_size, size=(n_atoms, 3))
        dists = [np.linalg.norm(pos[i] - pos[j])
                 for i in range(n_atoms) for j in range(i+1, n_atoms)]
        if min(dists) >= min_dist:
            return pos

# --- your code here ---

---
## Extension: Including temperature

In the simulations above, total energy is conserved and the system oscillates indefinitely. To model a system at finite **temperature** $T$, we couple the nuclei to a thermostat.

The simplest approach is **Langevin dynamics**, which adds friction and random kicks to Newton’s equation:

$$m\,\ddot{\mathbf{R}} = \mathbf{F}_{\text{elec}} - \gamma\, m\,\dot{\mathbf{R}} + \sqrt{2\gamma\, m\, k_B T}\;\boldsymbol{\eta}(t)$$

where $\gamma$ is a friction coefficient (in $1/\text{a.u. time}$) and $\boldsymbol{\eta}(t)$ is Gaussian white noise.

### Steps to implement

1. **Choose parameters**: target temperature $T$ (e.g. 300 K), friction $\gamma$ (e.g. $10^{-3}$ a.u.$^{-1}$). The Boltzmann constant in atomic units is $k_B = 3.1668 \times 10^{-6}$ Ha/K.

2. **Modify the Velocity Verlet step** using the BAOAB splitting scheme. After the standard velocity and position updates, apply a thermostat kick to the velocities:
   $$\mathbf{v} \leftarrow e^{-\gamma\,\Delta t}\,\mathbf{v} \;+\; \sqrt{\frac{k_B T}{m}\bigl(1 - e^{-2\gamma\,\Delta t}\bigr)}\;\boldsymbol{\xi}$$
   where $\boldsymbol{\xi} \sim \mathcal{N}(0,1)$.

3. **Run and observe**: total energy is no longer conserved, but the **kinetic temperature** $T_{\text{kin}} = \frac{2\langle E_k\rangle}{(3N-3)\,k_B}$ should fluctuate around the target $T$.

4. **Physical questions**: How does the equilibrium bond length change with temperature? At what temperature do the H$_2$ molecules dissociate?

In [ ]:
K_BOLTZMANN = 3.1668e-6  # Ha/K

# --- your code here ---